# JOUR 1 - EXPLORATION ET FEATURE ENGINEERING

**Projet Fil Rouge - Prédiction de Churn Client**

**Objectifs du Jour 1**:
- Explorer et comprendre le dataset
- Créer un pipeline de preprocessing robuste
- Générer au minimum 10 nouvelles features avancées
- Établir une baseline de référence

**Durée estimée**: 7 heures

## 0. SETUP ET IMPORTS

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, recall_score)

import warnings
warnings.filterwarnings('ignore')


sns.set_palette('husl')
%matplotlib inline

## **PARTIE 1.1** - EXPLORATION APPROFONDIE (2h)

### **Mission 1.1.1** - Chargement et Première Vue

In [ ]:
df1 = pd.read_csv("../data/archive/churn-bigml-80.csv")
df2 = pd.read_csv("../data/archive/churn-bigml-20.csv")

df = pd.concat([df1, df2], axis=0, ignore_index=True)

print(f"Shape du dataset: {df.shape}")
df.head()

Shape du dataset: (3333, 20)


,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   3333 non-null   object 
 1   Account length          3333 non-null   int64  
 2   Area code               3333 non-null   int64  
 3   International plan      3333 non-null   object 
 4   Voice mail plan         3333 non-null   object 
 5   Number vmail messages   3333 non-null   int64  
 6   Total day minutes       3333 non-null   float64
 7   Total day calls         3333 non-null   int64  
 8   Total day charge        3333 non-null   float64
 9   Total eve minutes       3333 non-null   float64
 10  Total eve calls         3333 non-null   int64  
 11  Total eve charge        3333 non-null   float64
 12  Total night minutes     3333 non-null   float64
 13  Total night calls       3333 non-null   int64  
 14  Total night charge      3333 non-null   

In [ ]:
df.describe()

,Account length,Area code,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls
count,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000
mean,101.064806,437.182418,8.099010,179.775098,100.435644,30.562307,200.980348,100.114311,17.083540,200.872037,100.107711,9.039325,10.237294,4.479448,2.764581,1.562856
std,39.822106,42.371290,13.688365,54.467389,20.069084,9.259435,50.713844,19.922625,4.310668,50.573847,19.568609,2.275873,2.791840,2.461214,0.753773,1.315491
min,1.000000,408.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,23.200000,33.000000,1.040000,0.000000,0.000000,0.000000,0.000000
25%,74.000000,408.000000,0.000000,143.700000,87.000000,24.430000,166.600000,87.000000,14.160000,167.000000,87.000000,7.520000,8.500000,3.000000,2.300000,1.000000
50%,101.000000,415.000000,0.000000,179.400000,101.000000,30.500000,201.400000,100.000000,17.120000,201.200000,100.000000,9.050000,10.300000,4.000000,2.780000,1.000000
75%,127.000000,510.000000,20.000000,216.400000,114.000000,36.790000,235.300000,114.000000,20.000000,235.300000,113.000000,10.590000,12.100000,6.000000,3.270000,2.000000
max,243.000000,510.000000,51.000000,350.800000,165.000000,59.640000,363.700000,170.000000,30.910000,395.000000,175.000000,17.770000,20.000000,20.000000,5.400000,9.000000


### **Mission 1.1.2** - Analyse de la Target

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_percent = df['Churn'].value_counts(normalize=True) * 100


labels = ['False', 'True']
values = [churn_counts.get(False, 0), churn_counts.get(True, 0)]
percents = [churn_percent.get(False, 0), churn_percent.get(True, 0)]
colors = ['#FF4500', '#00FF7F']

texts = [f"{v} ({p:.1f}%)" for v, p in zip(values, percents)]
fig = go.Figure()

for lbl, val, txt, color in zip(labels, values, texts, colors):
    for i, alpha in zip([0.1, 0.2, 0.3], [0.25, 0.15, 0.1]):
        fig.add_trace(go.Bar(
            x=[lbl],
            y=[val],
            marker=dict(color=color),
            width=0.5 + i,
            opacity=alpha,
            showlegend=False,
            hoverinfo='skip'
        ))
    fig.add_trace(go.Bar(
        x=[lbl],
        y=[val],
        text=[txt],
        textposition='inside',
        textfont=dict(color='white', size=16),
        marker=dict(color=color),
        width=0.4,
        name=lbl
    ))

fig.update_layout(
    title="Répartition du Churn",
    title_font=dict(size=24, color='white', family="Arial Black"),
    plot_bgcolor='#0B0C10',
    paper_bgcolor='#0B0C10',
    xaxis=dict(
        title='Churn',
        tickfont=dict(color='white', size=14),
        showgrid=False,
        zeroline=False
    ),
    yaxis=dict(
        title='Nombre de clients',
        tickfont=dict(color='white', size=14),
        showgrid=False,
        zeroline=False
    ),
    barmode='overlay'
)

fig.show()

**Question**: Le dataset est-il déséquilibré ? Quel est le ratio ?

**Réponse**: Oui le dataset est déséquilibré. Une classe à 2278 et une autre à 388, 6 fois plus de false que de true

### **Mission 1.1.3** - Valeurs Manquantes et Qualité des Données

In [17]:
n = len(df)

valeurs_manquantes = df.isna().mean() * 100
valeurs_uniques = df.nunique(dropna=False) / n * 100
valeurs_dominantes = df.apply(
    lambda col: col.value_counts(dropna=False, normalize=True).iloc[0] * 100
)
n_duplique = df.duplicated().sum()


summary = pd.DataFrame({
    "valeurs_manquantes": valeurs_manquantes,
    "valeurs_uniques": valeurs_uniques,
    "n_duplique": n_duplique,
}).sort_values(by="valeurs_manquantes", ascending=False)

print(summary)

                        valeurs_manquantes  valeurs_uniques  n_duplique
State                                  0.0         1.530153           0
Account length                         0.0         6.360636           0
Area code                              0.0         0.090009           0
International plan                     0.0         0.060006           0
Voice mail plan                        0.0         0.060006           0
Number vmail messages                  0.0         1.380138           0
Total day minutes                      0.0        50.015002           0
Total day calls                        0.0         3.570357           0
Total day charge                       0.0        50.015002           0
Total eve minutes                      0.0        48.334833           0
Total eve calls                        0.0         3.690369           0
Total eve charge                       0.0        43.204320           0
Total night minutes                    0.0        47.734773     

### **Mission 1.1.4** - Analyse Univariée

In [23]:
numerical_cols = [
    'Account length',
    'Number vmail messages',
    'Total day minutes', 'Total day calls', 'Total day charge',
    'Total eve minutes', 'Total eve calls', 'Total eve charge',
    'Total night minutes', 'Total night calls', 'Total night charge',
    'Total intl minutes', 'Total intl calls', 'Total intl charge',
    'Customer service calls'
]

colors = ['#FF4500', '#00FF7F', '#00FFFF', '#BF00FF']

rows = int(np.ceil(len(numerical_cols) / 3))
fig = make_subplots(rows=rows, cols=3, subplot_titles=numerical_cols)

for i, col in enumerate(numerical_cols):
    r = i // 3 + 1
    c = i % 3 + 1
    color = colors[i % len(colors)]
    
    fig.add_trace(
        go.Histogram(
            x=df[col],
            nbinsx=30,
            marker=dict(color=color),
            opacity=0.15,
            showlegend=False
        ),
        row=r, col=c
    )
    
    fig.add_trace(
        go.Histogram(
            x=df[col],
            nbinsx=30,
            marker=dict(color=color),
            opacity=0.9,
            showlegend=False
        ),
        row=r, col=c
    )

fig.update_layout(
    height=300 * rows,
    width=1200,
    title="Distribution des variables numériques",
    
    plot_bgcolor="#0B0C10",
    paper_bgcolor="#0B0C10",
    
    font=dict(color="white"),
    
    bargap=0.2
)

fig.update_xaxes(showgrid=False, color='lightgray')
fig.update_yaxes(showgrid=False, color='lightgray')

fig.show()

In [8]:
categorical_cols = [
    'State',
    'Area code',    
    'International plan',
    'Voice mail plan',
    'Churn'
]


colors = ['#D84315', '#008F39', '#005A92', '#6A1B9A']

for i, col in enumerate(categorical_cols):
    data = df[col].value_counts().reset_index()
    data.columns = [col, 'count']
    total = data['count'].sum()
    data['percentage'] = (data['count'] / total * 100).round(1)
    
    color = colors[i % len(colors)]
    fig = go.Figure()

    if col == 'State':
        data = data.sort_values('count', ascending=True)
        
        fig.add_trace(go.Bar(
            y=data[col],
            x=data['count'],
            orientation='h',
            marker=dict(
                color=color,
                line=dict(color='rgba(255,255,255,0.1)', width=1) 
            ),
            text=[f"{c} ({p}%)" for c, p in zip(data['count'], data['percentage'])],
            textposition='outside',
            textfont=dict(color='white', size=10),
            showlegend=False
        ))
        fig.update_layout(height=1000)

    
    elif col in ['International plan', 'Voice mail plan', 'Churn']:
        for idx, row in data.iterrows():
           
            fig.add_trace(go.Bar(
                name=f"{row[col]} ({row['percentage']}%)",
                x=[row[col]],
                y=[row['count']],
                marker=dict(
                    color=color, 
                    opacity=1 - (idx * 0.3), 
                    line=dict(color='white', width=0.5)
                ), 
                width=0.4,
                text=f"{row['count']}<br>({row['percentage']}%)",
                textposition='inside',
                textfont=dict(color='white')
            ))
        fig.update_layout(showlegend=True, legend=dict(font=dict(size=12, color="white")))
    
    
    else:
        fig.add_trace(go.Bar(
            x=data[col],
            y=data['count'],
            marker=dict(color=color),
            width=0.6,
            text=[f"{c}<br>({p}%)" for c, p in zip(data['count'], data['percentage'])],
            textposition='inside',
            textfont=dict(color='white', size=12),
            showlegend=False
        ))

    
    fig.update_layout(
        title=f"Distribution de {col}",
        plot_bgcolor="#0B0C10", 
        paper_bgcolor="#0B0C10",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False, 
            color='lightgray',
            type='category' if col != 'State' else 'linear'
        ),
        yaxis=dict(
            showgrid=False, 
            color='lightgray',
            type='category' if col == 'State' else 'linear',
            title="Nombre de clients" if col != 'State' else ""
        ),
        bargap=0.2
    )
    
    fig.show()

### Mission 1.1.5 - Analyse Bivariée

In [26]:
categorical_cols = [
    'State',
    'Area code',
    'International plan',
    'Voice mail plan'
]

colors = ['#FF4500', '#00FF7F', '#00FFFF', '#BF00FF']

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"Churn Rate - {col}" for col in categorical_cols]
)

for i, col in enumerate(categorical_cols):
    
    row = i // 2 + 1
    col_pos = i % 2 + 1
    
    churn_rate = df.groupby(col)['Churn'].mean() * 100
    churn_rate = churn_rate.sort_values(ascending=False)
    
    if len(churn_rate) > 10:
        churn_rate = churn_rate.head(10)
    
    x = churn_rate.index.astype(str)
    y = churn_rate.values
    
    color = colors[i % len(colors)]
    
    fig.add_trace(go.Bar(
        x=x,
        y=y,
        marker=dict(color=color),
        opacity=0.05,
        width=0.9,
        showlegend=False,
        hoverinfo='skip'
    ), row=row, col=col_pos)
    
    fig.add_trace(go.Bar(
        x=x,
        y=y,
        marker=dict(color=color),
        opacity=0.12,
        width=0.7,
        showlegend=False,
        hoverinfo='skip'
    ), row=row, col=col_pos)
    
    fig.add_trace(go.Bar(
        x=x,
        y=y,
        marker=dict(color=color),
        opacity=0.25,
        width=0.55,
        showlegend=False,
        hoverinfo='skip'
    ), row=row, col=col_pos)
    
    fig.add_trace(go.Bar(
        x=x,
        y=y,
        marker=dict(
            color=color,
            line=dict(color='white', width=0.5)
        ),
        width=0.4,
        text=[f"{v:.1f}%" for v in y],
        textposition='inside',
        textfont=dict(color='white'),
        showlegend=False
    ), row=row, col=col_pos)

fig.update_layout(
    height=800,
    width=1100,
    title="Taux de Churn par Variable (Dashboard Néon)",
    
    plot_bgcolor="#0B0C10",
    paper_bgcolor="#0B0C10",
    
    font=dict(color="white"),
    
    bargap=0.2
)

fig.update_xaxes(showgrid=False, tickangle=45, color='lightgray')
fig.update_yaxes(showgrid=False, color='lightgray', title="Taux (%)")

fig.show()

In [27]:
df['Churn_binary'] = df['Churn'].map({False: 0, True: 1})

numerical_cols = [
    'Account length',
    'Number vmail messages',
    'Total day minutes', 'Total day calls', 'Total day charge',
    'Total eve minutes', 'Total eve calls', 'Total eve charge',
    'Total night minutes', 'Total night calls', 'Total night charge',
    'Total intl minutes', 'Total intl calls', 'Total intl charge',
    'Customer service calls',
    'Churn_binary'
]

corr_matrix = df[numerical_cols].corr()
corr_with_churn = corr_matrix['Churn_binary'].sort_values(ascending=False)

corr_data = corr_with_churn.drop('Churn_binary')
cols = corr_data.index
vals = corr_data.values

colors = ['#FF4500' if v > 0 else '#00FFFF' for v in vals]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=cols,
    y=vals,
    marker=dict(color=colors),
    opacity=0.2,
    width=0.6,
    showlegend=False
))

fig.add_trace(go.Bar(
    x=cols,
    y=vals,
    marker=dict(color=colors),
    width=0.4,
    text=[f"{v:.2f}" for v in vals],
    textposition='inside',
    textfont=dict(color='white'),
    showlegend=False
))

fig.update_layout(
    title="Corrélation des variables numériques avec Churn (Dark / Néon)",
    plot_bgcolor="#0B0C10",
    paper_bgcolor="#0B0C10",
    font=dict(color="white"),
    xaxis=dict(showgrid=False, tickangle=45, color='white'),
    yaxis=dict(showgrid=False, color='white')
)

fig.show()

df['Churn_binary'] = df['Churn'].map({False: 0, True: 1})

numerical_cols = [
    'Account length',
    'Number vmail messages',
    'Total day minutes', 'Total day calls', 'Total day charge',
    'Total eve minutes', 'Total eve calls', 'Total eve charge',
    'Total night minutes', 'Total night calls', 'Total night charge',
    'Total intl minutes', 'Total intl calls', 'Total intl charge',
    'Customer service calls',
    'Churn_binary'
]

corr_matrix = df[numerical_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmin=-1,
    zmax=1,
    text=np.round(corr_matrix.values, 2),
    texttemplate="%{text}",
    colorbar=dict(title="Corrélation")
))

fig.update_layout(
    title="Matrice de corrélation (toutes variables numériques)",
    plot_bgcolor="#0B0C10",
    paper_bgcolor="#0B0C10",
    font=dict(color="white"),
    xaxis=dict(tickangle=45),
)

fig.show()



print("Colonnes parfaitement corrélées (corr = 1) :\n")

found = False

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        
        if np.isclose(corr_value, 1.0):
            print(f"{corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]} : {corr_value}")
            found = True

if not found:
    print("Aucune corrélation parfaite trouvée.")

Colonnes parfaitement corrélées (corr = 1) :

Total day minutes ↔ Total day charge : 0.9999999521904008
Total eve minutes ↔ Total eve charge : 0.9999997760198508
Total night minutes ↔ Total night charge : 0.9999992148758784
Total intl minutes ↔ Total intl charge : 0.9999927417510344


In [28]:
df['Total_charge'] = (
    df['Total day charge'] +
    df['Total eve charge'] +
    df['Total night charge'] +
    df['Total intl charge']
)

churned = df[df['Churn'] == True]

avg_churn = churned['Total_charge'].sum().mean()

print(f"Revenu mensuel moyen (clients churn) : {avg_churn:.2f}")

Revenu mensuel moyen (clients churn) : 31566.93


### Mission 1.1.6 - Insights Métier

**TODO**: À partir de votre exploration, répondez aux questions suivantes:

1. **Quels sont les 3 profils clients les plus à risque de churn ?**
   Il y a une correlation clair entre des variables numériques et le churn
   - En 1er on retrouve les personnes qui font des appelles au services clients 
   - En 2eme les gens qui payent leurs appels le plus cher par jour
   - En 3eme les gens qui ont un temps d'appel élevé par jour

2. **Quel est l'impact financier mensuel moyen du churn ?**
   - I mpact financier mensuel moyen du churn : 31566.93$

3. **Formulez 3 hypothèses métier à tester:**
   - H1: Plus le prix est chère plus il y a du churn.
   - H2: Le nombre d’appels au service client influence le churn.
   - H3: améliorer le service client, pourrait augmenter le taux de churn.

## PARTIE 1.2 - FEATURE ENGINEERING AVANCÉ (3h)

### Mission 1.2.1 - Créer au minimum 10 nouvelles features

#### A. Features de Ratio

In [12]:
# TODO: Créer au moins 3 features de ratio
# Exemples:
# - Dépense moyenne mensuelle: TotalCharges / tenure
# - Ratio MonthlyCharges / nombre de services

df['AvgMonthlySpend'] = ...
df['ChargesPerTenure'] = ...
# ...


#### B. Features d'Agrégation

In [13]:
# TODO: Créer au moins 2 features d'agrégation
# Exemples:
# - Nombre total de services souscrits
# - Score d'engagement (tenure × services)

service_cols = ['PhoneService', 'InternetService', 'OnlineSecurity', 
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                'StreamingTV', 'StreamingMovies']

df['TotalServices'] = ...
df['EngagementScore'] = ...


#### C. Features d'Interaction

In [14]:
# TODO: Créer au moins 2 features d'interaction
# Exemples:
# - Contract × InternetService
# - tenure × MonthlyCharges

...

Ellipsis

#### D. Features Binaires

In [15]:
# TODO: Créer au moins 2 features binaires
# Exemples:
# - IsNewCustomer (tenure < 6 mois)
# - IsPremium (MonthlyCharges > 80)

df['IsNewCustomer'] = ...
df['IsPremium'] = ...


#### E. Transformations Mathématiques (Optionnel)

In [16]:
# TODO: Si pertinent, créer des transformations logarithmiques ou polynomiales
# Exemples:
# - Log des charges
# - tenure au carré

...

Ellipsis

#### F. Vérification des Features Créées

In [ ]:
# TODO: Lister toutes les nouvelles features créées
new_features = [...]  # Minimum 10

print(f"Nombre de nouvelles features: {len(new_features)}")
print("Features créées:", new_features)

# Afficher les premières lignes
df[new_features].head()

### Mission 1.2.2 - Pipeline de Preprocessing

In [ ]:
# TODO: Préparer les données pour la modélisation

# 1. Supprimer customerID (non informatif)
df = ...

# 2. Séparer features et target
X = ...
y = ...

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# TODO: Encodage des variables catégorielles
# Méthode: pd.get_dummies() ou OneHotEncoder

X_encoded = ...

print(f"Shape après encodage: {X_encoded.shape}")

## PARTIE 1.3 - BASELINE ET VALIDATION (2h)

### Mission 1.3.1 - Train/Test Split

In [ ]:
# TODO: Split stratifié 80/20
X_train, X_test, y_train, y_test = ...

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}")
print(f"Test churn rate: {y_test.mean():.2%}")

In [ ]:
# TODO: Normalisation des features numériques
scaler = ...
...


### Mission 1.3.2 - Modèle Baseline (Logistic Regression)

In [ ]:
# TODO: Entraîner une Logistic Regression simple
lr_baseline = ...
...

# Prédictions
y_pred = ...
y_proba = ...


In [ ]:
# TODO: Évaluer les performances
print("Classification Report:")
...

print(f"\nROC-AUC: {roc_auc_score(...)}")


In [ ]:
# TODO: Matrice de confusion
...

### Mission 1.3.3 - Cross-Validation

In [ ]:
# TODO: StratifiedKFold avec 5 folds
from sklearn.model_selection import cross_val_score

skf = ...
cv_scores = ...

print(f"CV Recall: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## SAUVEGARDE DES DONNÉES PRÉPARÉES

In [ ]:
# TODO: Sauvegarder le dataset enrichi et les splits
# df.to_csv('../data/churn_engineered.csv', index=False)
# joblib.dump([X_train, X_test, y_train, y_test], '../data/train_test_splits.pkl')

...

## CONCLUSION JOUR 1

**Résumé des réalisations**:
- Dataset exploré: ... lignes, ... colonnes
- Features créées: ... (minimum 10)
- Baseline Logistic Regression:
  - Recall: ...
  - Precision: ...
  - ROC-AUC: ...

**Prochaines étapes (Jour 2)**:
- Benchmark de 5-6 modèles
- Gestion du déséquilibre (SMOTE, class_weight)
- Optimisation des hyperparamètres